# Preprocessing a subset of the realistic Dataset
Especially labeling and checking which trajectories should be used will be done here

In [12]:
import numpy as np
import pandas as pd
import sys
import shutil
import os

In [21]:
def angle_to_axis(vx, vy, vz, zhat):
    norm = np.sqrt(vx*vx + vy*vy + vz*vz)
    if norm == 0:
        return np.nan
    # Skalarprodukt durch Norm => cos(theta)
    cos_theta = (vx*zhat[0] + vy*zhat[1] + vz*zhat[2]) / norm
    # numerische Stabilisierung
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return np.degrees(np.arccos(cos_theta))

In [22]:
# load the dataframe
df = pd.read_csv("/data/lkolmar/datasets/realistic/config/simulation.csv")
df = df[df["finished"] == True]
print(f"Loaded {len(df)} finished simulations")

Loaded 2874 finished simulations


In [23]:
valid_labels = []
threshold = 20  # degrees
for i in range(len(df)):
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    pos_x = angle_to_axis(vx, vy, vz, (1, 0, 1))  # backspin
    neg_x = angle_to_axis(vx, vy, vz, (-1, 0, 0)) # topspin

    if min(pos_x, neg_x) < threshold:
        valid_labels.append(i)

print(f"Found {len(valid_labels)} valid labels out of {len(df)}")

Found 407 valid labels out of 2874


In [24]:
backspin = 0
topspin = 0
rpss = []
high = 0
mid = 0
low = 0
for i in valid_labels:
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    if vx > 0:
        label = "backspin"
        backspin += 1
    else:
        label = "topspin"
        topspin += 1

    path = "/data/lkolmar/datasets/realistic/" + sample["path"]
    metadata = pd.read_csv(path + "metadata.csv")
    rps = metadata["rotation_omega"].values[0]
    rpss.append(rps)
    print(f"Index {i}: {label}, {rps:.1f}")

    if rps >= 100:
        high += 1
    elif rps >= 70:
        mid += 1
    else:
        low += 1

print(f"Backspin: {backspin}, Topspin: {topspin}, Mean RPS: {np.mean(rpss):.1f} +/- {np.std(rpss):.1f}")
print(f"Max rps: {np.max(rpss):.1f}, Min rps: {np.min(rpss):.1f}")
print(f"High (>100 rps): {high}, Mid (70-100 rps): {mid}, Low (<70 rps): {low}")

Index 460: backspin, 139.5
Index 474: backspin, 139.5
Index 832: backspin, 132.6
Index 852: backspin, 120.9
Index 853: backspin, 127.6
Index 854: backspin, 134.7
Index 872: backspin, 112.1
Index 873: backspin, 117.8
Index 874: backspin, 123.9
Index 875: backspin, 130.5
Index 876: backspin, 137.5
Index 892: backspin, 111.2
Index 893: backspin, 116.2
Index 894: backspin, 121.7
Index 895: backspin, 127.6
Index 896: backspin, 134.0
Index 944: backspin, 133.3
Index 965: backspin, 112.1
Index 966: backspin, 119.3
Index 967: backspin, 126.9
Index 968: backspin, 134.7
Index 987: backspin, 101.6
Index 988: backspin, 107.8
Index 989: backspin, 114.5
Index 990: backspin, 121.7
Index 991: backspin, 129.1
Index 992: backspin, 136.8
Index 1010: backspin, 99.8
Index 1011: backspin, 105.2
Index 1012: backspin, 111.2
Index 1013: backspin, 117.8
Index 1014: backspin, 124.7
Index 1015: backspin, 131.9
Index 1016: backspin, 139.5
Index 1032: backspin, 99.8
Index 1033: backspin, 104.3
Index 1034: backspin,

In [25]:
label_names = {
    "topspin_slow": 0,
    "topspin_mid": 1,
    "topspin_fast": 2,
    "backspin_slow": 3,
    "backspin_mid": 4,
    "backspin_fast": 5,
}

# Select valid balls and copy to new subset dataset

In [ ]:
new_index = 0
labels = []
for i in valid_labels:
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    path = "/data/lkolmar/datasets/realistic/" + sample["path"]
    metadata = pd.read_csv(path + "metadata.csv")
    rps = metadata["rotation_omega"].values[0]

    if vx > 0:
        label = "backspin"
        if rps >= 100:
            labels.append(label_names["backspin_fast"])
        elif rps >= 70:
            labels.append(label_names["backspin_mid"])
        else:
            labels.append(label_names["backspin_slow"])

    else:
        label = "topspin"
        if rps >= 100:
            labels.append(label_names["topspin_fast"])
        elif rps >= 70:
            labels.append(label_names["topspin_mid"])
        else:
            labels.append(label_names["topspin_slow"])

    index_str = str(new_index).zfill(5)
    os.makedirs(f"/data/lkolmar/datasets/subset/preprocessed/{index_str}", exist_ok=True)
    new_path = f"/data/lkolmar/datasets/subset/preprocessed/{index_str}/{index_str}_roi.hdf5"
    # print(path[-6:-1])
    shutil.copyfile(f"/data/lkolmar/datasets/realistic/preprocessed/{path[-6:-1]}/{path[-6:-1]}_roi.hdf5", new_path)
    new_index += 1
    # print(f"Copied {new_path} with label {label} and {rps:.1f} rps")
    # if new_index >= 10:
    # break

df_labels = pd.DataFrame({'index': range(len(labels)), 'label': labels})
print(df_labels)

Copied /data/lkolmar/datasets/subset/preprocessed/00000/00000_roi.hdf5 with label backspin and 139.5 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00001/00001_roi.hdf5 with label backspin and 139.5 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00002/00002_roi.hdf5 with label backspin and 132.6 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00003/00003_roi.hdf5 with label backspin and 120.9 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00004/00004_roi.hdf5 with label backspin and 127.6 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00005/00005_roi.hdf5 with label backspin and 134.7 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00006/00006_roi.hdf5 with label backspin and 112.1 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00007/00007_roi.hdf5 with label backspin and 117.8 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00008/00008_roi.hdf5 with label backspin and 123.9 rps
Copied /data/lkolmar/datasets/subset/preprocessed/00009

In [29]:
df_labels.to_csv("/data/lkolmar/datasets/subset/config/labels.csv", index=False)